<a href="https://colab.research.google.com/github/azcsprof/UCLA-XL161/blob/Final-Project-Starter-Code/XL161_Final_Project_Option_1_Starter_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COM SCI XL161 FINAL PROJECT — OPTION 1
# Intelligent Navigation Agent
# ============================================================
#
# Core idea:
# This project begins with a search-based agent, like the midterm,
# but extends it toward Modules 6–9.
#
# The agent must move through a grid world from START to GOAL.
# It must reason about:
#
# Module 1: agents, states, actions, goals
# Module 2: state-space search
# Module 3: heuristic search / A*
# Module 4: constraints and invalid moves
# Module 6: rule-based reasoning
# Module 8: uncertainty / probability
# Module 9: simple learning from experience
#
# Students should NOT treat this as just a pathfinding program.
# The important question is:
#
#     How does the system decide what to do?
#
# ============================================================


import heapq


# ============================================================
# 1. GRID SYMBOLS
# ============================================================
#
# S = start position
# G = goal position
# # = wall / blocked cell
# . = open path
# H = possible hazard
#
# The agent can move up, down, left, or right.
#
# This grid is the agent's "environment."
# The agent does not think like a human.
# It only reasons using the representation we give it.
#
# ============================================================

grid = [
    ["S", ".", ".", "#", ".", "."],
    [".", "#", ".", "#", "H", "."],
    [".", "#", ".", ".", ".", "."],
    [".", ".", ".", "#", "#", "."],
    ["H", "#", ".", ".", ".", "G"]
]


# ============================================================
# 2. START AND GOAL LOCATIONS
# ============================================================
#
# These positions define the problem.
#
# A "state" is simply the agent's current location.
# For this project, a state is represented as a tuple:
#
#     (row, column)
#
# Example:
#     (0, 0) means row 0, column 0
#
# ============================================================

START = (0, 0)
GOAL = (4, 5)


# ============================================================
# 3. HAZARD PROBABILITIES
# ============================================================
#
# Module 8 connection:
#
# Some cells are uncertain.
# The agent does not know for sure whether a hazard is dangerous.
# Instead, each hazard cell has a probability.
#
# A probability close to 1 means high danger.
# A probability close to 0 means low danger.
#
# This allows the agent to make decisions under uncertainty instead
# of treating every cell as simply safe or unsafe.
#
# ============================================================

hazard_probability = {
    (1, 4): 0.70,
    (4, 0): 0.40
}


# ============================================================
# 4. LEARNED PENALTIES
# ============================================================
#
# Module 9 connection:
#
# The agent can learn from experience.
#
# If the agent travels through a risky cell and receives negative
# feedback, it can increase the cost of that cell in the future.
#
# This is not advanced machine learning.
# It is a simple learning mechanism:
#
#     bad experience → higher penalty → less likely future path
#
# ============================================================

learned_penalties = {
    (1, 4): 0,
    (4, 0): 0
}


# ============================================================
# 5. DISPLAY THE GRID
# ============================================================
#
# This function prints the grid so we can see the environment.
#
# The agent itself does not need this function to reason.
# This is for humans reading the output.
#
# ============================================================

def print_grid():
    print("\nGRID WORLD:")
    for row in grid:
        print(" ".join(row))
    print()


# ============================================================
# 6. CHECK WHETHER A STATE IS VALID
# ============================================================
#
# Module 4 connection:
#
# A constraint tells the agent what is NOT allowed.
#
# Here, a move is invalid if:
#   1. It goes outside the grid
#   2. It hits a wall
#
# This is constraint-based reasoning because the system eliminates
# actions that violate the rules of the environment.
#
# ============================================================

def is_valid_state(state):
    row, col = state

    # Constraint 1: the agent must stay inside the grid.
    if row < 0 or row >= len(grid):
        return False

    if col < 0 or col >= len(grid[0]):
        return False

    # Constraint 2: the agent cannot move through walls.
    if grid[row][col] == "#":
        return False

    return True


# ============================================================
# 7. GENERATE POSSIBLE ACTIONS
# ============================================================
#
# Module 1 and Module 2 connection:
#
# An agent needs actions.
# In this environment, the possible actions are movement directions.
#
# From a current state, the system generates neighboring states.
# Search works by exploring these possible future states.
#
# ============================================================

def get_neighbors(state):
    row, col = state

    possible_moves = [
        (row - 1, col),  # move up
        (row + 1, col),  # move down
        (row, col - 1),  # move left
        (row, col + 1)   # move right
    ]

    valid_neighbors = []

    for next_state in possible_moves:
        if is_valid_state(next_state):
            valid_neighbors.append(next_state)

    return valid_neighbors


# ============================================================
# 8. HEURISTIC FUNCTION
# ============================================================
#
# Module 3 connection:
#
# A* search uses a heuristic to estimate distance to the goal.
#
# This function uses Manhattan distance:
#
#     distance = vertical distance + horizontal distance
#
# The heuristic does not guarantee the actual best path by itself.
# It only gives the agent a way to prioritize promising paths.
#
# ============================================================

def heuristic(state, goal):
    row, col = state
    goal_row, goal_col = goal

    return abs(row - goal_row) + abs(col - goal_col)


# ============================================================
# 9. RULE-BASED REASONING
# ============================================================
#
# Module 6 connection:
#
# The agent uses simple IF/THEN rules to reason about danger.
#
# Example:
#
# IF a cell has hazard probability greater than 0.60,
# THEN treat it as high risk.
#
# These rules do not "learn."
# They apply fixed logic to the current representation.
#
# ============================================================

def rule_based_risk_check(state):
    probability = hazard_probability.get(state, 0)

    if probability >= 0.60:
        return "HIGH_RISK"

    if probability > 0:
        return "LOW_RISK"

    return "SAFE"


# ============================================================
# 10. COST FUNCTION
# ============================================================
#
# This is where multiple course ideas come together.
#
# The cost of moving into a cell depends on:
#
#   1. Basic movement cost
#   2. Rule-based risk category
#   3. Probability of hazard
#   4. Learned penalty from past experience
#
# This matters because A* does not just ask:
#
#     Can I move there?
#
# It asks:
#
#     How costly is it to move there?
#
# ============================================================

def movement_cost(state):
    base_cost = 1

    risk_category = rule_based_risk_check(state)
    probability = hazard_probability.get(state, 0)
    learned_cost = learned_penalties.get(state, 0)

    # High-risk cells receive a larger penalty.
    if risk_category == "HIGH_RISK":
        risk_penalty = 5

    # Low-risk cells receive a smaller penalty.
    elif risk_category == "LOW_RISK":
        risk_penalty = 2

    # Safe cells receive no risk penalty.
    else:
        risk_penalty = 0

    # Probability adds another uncertainty-based cost.
    probability_penalty = probability * 3

    total_cost = base_cost + risk_penalty + probability_penalty + learned_cost

    return total_cost


# ============================================================
# 11. A* SEARCH
# ============================================================
#
# Modules 2 and 3 connection:
#
# A* combines:
#
#   g(n): cost so far
#   h(n): estimated distance to goal
#
# The agent selects the next state using:
#
#   f(n) = g(n) + h(n)
#
# In this version, g(n) includes hazard costs and learned penalties.
# That means the agent is not just searching for the shortest path.
# It is searching for a path that balances distance and risk.
#
# ============================================================

def a_star_search(start, goal):
    frontier = []

    # Each item in the frontier stores:
    # (priority, current_state)
    heapq.heappush(frontier, (0, start))

    came_from = {}
    cost_so_far = {}

    came_from[start] = None
    cost_so_far[start] = 0

    print("\n=== SEARCH TRACE ===")

    while frontier:
        current_priority, current_state = heapq.heappop(frontier)

        print(f"\nExpanding state: {current_state}")
        print(f"Current cost so far: {cost_so_far[current_state]:.2f}")

        if current_state == goal:
            print("\nGoal reached.")
            break

        for next_state in get_neighbors(current_state):
            step_cost = movement_cost(next_state)
            new_cost = cost_so_far[current_state] + step_cost

            print(f"  Considering move to {next_state}")
            print(f"    Risk category: {rule_based_risk_check(next_state)}")
            print(f"    Movement cost: {step_cost:.2f}")

            if next_state not in cost_so_far or new_cost < cost_so_far[next_state]:
                cost_so_far[next_state] = new_cost

                priority = new_cost + heuristic(next_state, goal)

                heapq.heappush(frontier, (priority, next_state))
                came_from[next_state] = current_state

                print(f"    Updated best path to {next_state}")
                print(f"    Priority: {priority:.2f}")

    return came_from, cost_so_far


# ============================================================
# 12. RECONSTRUCT PATH
# ============================================================
#
# After search finishes, this function traces backward from the goal
# to the start.
#
# This lets us see the final decision path chosen by the agent.
#
# ============================================================

def reconstruct_path(came_from, start, goal):
    if goal not in came_from:
        return []

    current = goal
    path = []

    while current != start:
        path.append(current)
        current = came_from[current]

    path.append(start)
    path.reverse()

    return path


# ============================================================
# 13. SIMPLE LEARNING UPDATE
# ============================================================
#
# Module 9 connection:
#
# After the agent completes a path, we simulate feedback.
#
# If the path included a hazard cell, the agent increases the learned
# penalty for that cell.
#
# This means future searches may avoid that cell.
#
# Again, this is intentionally simple.
# The goal is not advanced ML.
# The goal is to show how feedback can change future behavior.
#
# ============================================================

def update_learning_from_path(path):
    print("\n=== LEARNING UPDATE ===")

    for state in path:
        if state in hazard_probability:
            learned_penalties[state] += 2

            print(f"Agent received negative feedback for {state}.")
            print(f"New learned penalty for {state}: {learned_penalties[state]}")


# ============================================================
# 14. RUN ONE FULL AGENT TRIAL
# ============================================================
#
# This function runs the full system:
#
#   1. Print the environment
#   2. Search for a path
#   3. Display the path
#   4. Apply learning feedback
#
# Students can run this multiple times and observe whether behavior
# changes after learning penalties are updated.
#
# ============================================================

def run_agent_trial():
    print_grid()

    came_from, cost_so_far = a_star_search(START, GOAL)

    path = reconstruct_path(came_from, START, GOAL)

    print("\n=== FINAL PATH ===")
    if path:
        print(path)
        print(f"Total path cost: {cost_so_far[GOAL]:.2f}")
    else:
        print("No path found.")

    update_learning_from_path(path)


# ============================================================
# 15. MAIN PROGRAM
# ============================================================
#
# Run the agent twice.
#
# The second run may behave differently if the first path caused
# learned penalties to increase.
#
# This allows students to observe how experience changes decisions.
#
# ============================================================

print("\n================================================")
print("INTELLIGENT NAVIGATION AGENT")
print("Modules 1–9 Integrated Starter Code")
print("================================================")

print("\nFIRST RUN:")
run_agent_trial()

print("\n\nSECOND RUN AFTER LEARNING:")
run_agent_trial()

# Student modification options:
# 1. Change the hazard probabilities.
# 2. Add a new hazard cell.
# 3. Add a new rule in rule_based_risk_check().
# 4. Change the heuristic.
# 5. Change how learning updates penalties.
# 6. Add a new constraint, such as battery limit or forbidden zone.